In [1]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus, urlparse

import pandas as pd
import psycopg


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".env").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing .env and src/.")


ROOT = find_project_root(Path.cwd().resolve())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ENV_PATH = ROOT / ".env"


def load_database_url() -> str:
    env_url = os.getenv("SUPABASE_DATABASE_URL")
    if env_url:
        return env_url

    if not ENV_PATH.exists():
        raise FileNotFoundError("No .env file found in the project root.")

    values = {}
    for raw_line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")

    if values.get("SUPABASE_DATABASE_URL"):
        return values["SUPABASE_DATABASE_URL"]

    host = values.get("SUPABASE_DB_HOST") or ""
    if not host:
        project_url = values.get("NEXT_PUBLIC_SUPABASE_URL")
        if project_url:
            parsed = urlparse(project_url)
            host = parsed.hostname or ""
            if host and not host.startswith("db."):
                host = f"db.{host}"

    password = values.get("SUPABASE_DB_PASSWORD")
    if not host or not password:
        raise ValueError("Set SUPABASE_DATABASE_URL or SUPABASE_DB_HOST + SUPABASE_DB_PASSWORD in .env.")

    user = values.get("SUPABASE_DB_USER") or "postgres"
    dbname = values.get("SUPABASE_DB_NAME") or "postgres"
    port = values.get("SUPABASE_DB_PORT") or "5432"
    return f"postgresql://{user}:{quote_plus(password)}@{host}:{port}/{dbname}?sslmode=require"


DATABASE_URL = load_database_url()
print("Database URL loaded successfully.")
print("Using host:", urlparse(DATABASE_URL).hostname)


Database URL loaded successfully.
Using host: aws-1-ap-northeast-1.pooler.supabase.com


In [2]:
SQL_QUERY = """
select * from departments where id = 1;
""".strip()

print("Query to test:")
print(SQL_QUERY)


Query to test:
select * from departments where id = 1;


In [3]:
import sys
from pathlib import Path

import psycopg


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".env").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing .env and src/.")


ROOT = find_project_root(Path.cwd().resolve())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nl2sql.validation import SQLValidationGatekeeper

safety = SQLValidationGatekeeper(database_url=None).validate(SQL_QUERY)
print("Safety status:", safety.status.value)
print("Safety message:", safety.user_message)
if not safety.ok:
    raise RuntimeError(f"Query safety check failed: {safety.user_message}")

try:
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.execute(SQL_QUERY)
            rows = cur.fetchall()
            columns = [desc[0] for desc in cur.description]
except Exception as exc:
    rows = []
    columns = []

    print("DB execution failed:", exc)

print("Columns:", columns)
print("Rows returned:", len(rows))

Safety status: ok
Safety message: SQL passed safety checks.
DB execution failed: column "id" does not exist
LINE 1: select * from departments where id = 1;
                                        ^
Columns: []
Rows returned: 0


In [4]:
df = pd.DataFrame(rows, columns=columns)
df


""


In [5]:
if df.empty:
    print("No rows were returned. Check the database connection or change the query.")
else:
    print("Real DB result confirmed: this query returned rows from the database.")


No rows were returned. Check the database connection or change the query.


In [6]:
import sys
from pathlib import Path

# Add the project's src/ directory to the path
project_root = Path.cwd().parent  # noteboooks/ -> project root
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [8]:
from nl2sql.engine import NL2SQLPromptComposer
from infastructure.llm.llm_client import GroqLLMClient
from nl2sql.validation import SQLValidationGatekeeper
from nl2sql.retry_engine import NL2SQLEngine, EngineStatus
from nl2sql.executor import SQLQueryExecutor

composer = NL2SQLPromptComposer.from_env(".env")
llm_client = GroqLLMClient(env_path=".env")
validator = SQLValidationGatekeeper.from_env(".env")
executor = SQLQueryExecutor.from_env(".env")

bundle = composer.compose("list down all doctors in cardiology department")

print("=" * 60)
print("SYSTEM PROMPT:")
print("=" * 60)
print(bundle.system_prompt)

print("\n" + "=" * 60)
print("USER PROMPT:")
print("=" * 60)
print(bundle.user_prompt)

print("\n" + "=" * 60)
print("SELECTED TABLES:", bundle.selected_tables)
print("=" * 60 + "\n")

engine = NL2SQLEngine(composer, llm_client, validator, max_retries=3)

result = engine.generate_sql("list down all doctors in cardiology department")

print(result.status)
print(result)

if result.status == EngineStatus.SUCCESS:
    exec_result = executor.execute(result.sql)
    if exec_result.success:
        for row in exec_result.rows:
            print(row)
    else:
        print(exec_result.message)
elif result.status == EngineStatus.CLARIFICATION_NEEDED:
    print("Clarification needed:", result.message)
else:
    print("Failed:", result.message)

SYSTEM PROMPT:
You are a schema-aware NL2SQL engine for MediCore Hospital. Generate a single valid PostgreSQL query that answers the user's question. Use only tables, columns, and joins present in the provided schema context. Never invent table names, column names, or relationships. Prefer explicit JOINs that follow the foreign keys. Return SQL only, with no markdown, no prose, and no explanations. If the question cannot be answered safely from the schema, return a short clarification question only.

USER PROMPT:
User question:
list down all doctors in cardiology department

Schema context:
Table departments:
Columns:
  - department_id (integer, not null, default=nextval('departments_department_id_seq'::regclass))
  - department_name (character varying, not null)
  - location (character varying, nullable)
  - phone (character varying, nullable)
  - head_of_department (character varying, nullable)
Sample rows:
  - {'department_id': 1, 'department_name': 'Cardiology', 'location': 'Buildi

In [12]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

In [19]:
from nl2sql.engine import NL2SQLPromptComposer
from infastructure.llm.llm_client import GroqLLMClient
from nl2sql.validation import SQLValidationGatekeeper
from nl2sql.retry_engine import NL2SQLEngine
from nl2sql.executor import SQLQueryExecutor

from agents.intent_router import IntentRouterAgent
from agents.sql_generator import SQLGeneratorAgent
from agents.result_interpreter import ResultInterpreterAgent
from orchestrator.orchestrator import Orchestrator
from tracing.tracing import AgentTracer

# Part 1 components
composer = NL2SQLPromptComposer.from_env(".env")
llm_client = GroqLLMClient(env_path=".env")
validator = SQLValidationGatekeeper.from_env(".env")
executor = SQLQueryExecutor.from_env(".env")
engine = NL2SQLEngine(composer, llm_client, validator, max_retries=3)

# Part 2 agents
router = IntentRouterAgent(llm_client)
sql_agent = SQLGeneratorAgent(engine, executor)
interpreter = ResultInterpreterAgent(llm_client)

orchestrator = Orchestrator(router, sql_agent, interpreter, tracer=AgentTracer("traces"))

result = orchestrator.run("Which doctors have the highest no-show rates?")

print(result.status)
print(result.summary)
print(result.chart_type)
for row in result.rows[:5]:
    print(row)

success
The top 10 doctors with the highest no-show rates have rates ranging from 13.10% to 14.67%, with Yasith Jayawardena having the highest rate at 14.67% and 55 no-shows out of 375 total appointments. The total no-show counts for these doctors range from 49 to 57, with Nadeeka Zoysa and Hashini Fernando both having 57 no-shows. These high no-show rates may indicate issues with patient engagement or scheduling practices.
bar
{'doctor_id': 329, 'first_name': 'Yasith', 'last_name': 'Jayawardena', 'no_show_count': 55, 'total_appointments': 375, 'no_show_rate': Decimal('14.67')}
{'doctor_id': 65, 'first_name': 'Nadeeka', 'last_name': 'Zoysa', 'no_show_count': 57, 'total_appointments': 404, 'no_show_rate': Decimal('14.11')}
{'doctor_id': 261, 'first_name': 'Hashini', 'last_name': 'Fernando', 'no_show_count': 57, 'total_appointments': 407, 'no_show_rate': Decimal('14.00')}
{'doctor_id': 32, 'first_name': 'Vishmi', 'last_name': 'Abeykoon', 'no_show_count': 51, 'total_appointments': 374, 'n